## Ex:2 Data Wrangling and Transformation 
### Objective

To perform data wrangling and transformation on a dataset using Python and Pandas by handling missing values, removing duplicates, correcting data types, filtering data, and transforming variables into a suitable format for data analysis and machine learning.



##  Dataset Description

The dataset contains information about **student academic performance, educational background, MBA specialization, and placement salary**.

###  Dataset Attributes

| Column | Description |
|:---|:---|
| `sl_no` | Serial number / student identifier |
| `gender` | Gender of the student |
| `hsc_p` | Higher Secondary / 12th percentage |
| `hsc_s` | Higher Secondary stream |
| `degree_p` | Undergraduate degree percentage |
| `degree_t` | Undergraduate degree type |
| `etest_p` | Employability / entrance test percentage |
| `specialisation` | MBA specialization |
| `mba_p` | MBA percentage |
| `salary` | Salary offered after placement |

---

## Experiment Question

 **Using the given student placement dataset, perform data wrangling and transformation by handling missing values, scaling numerical features, detecting and treating outliers, encoding categorical variables, and generating a final model-ready dataset.**




Data wrangling and transformation is the process of converting raw student placement data into a clean, consistent, and machine-learning-ready dataset. In this experiment, the given student placement dataset is first loaded into a Pandas DataFrame and explored by examining its rows, columns, data types, and descriptive statistics. Missing values are then identified and handled by removing records with missing `salary` values and replacing missing values in `hsc_p`, `degree_p`, and `etest_p` with their respective mean values. The numerical attributes such as `hsc_p`, `degree_p`, `etest_p`, and `salary` are transformed using feature-scaling techniques such as `StandardScaler` and `MinMaxScaler` to bring the variables into suitable numerical ranges. The dataset is then divided into input features (`X`) and the target variable (`Y`), where `salary` is considered the target. The preprocessed data is saved as `Pre.csv` for further processing. Next, possible outliers in the `salary` attribute are identified using a boxplot and statistically detected using the Z-score method, where values with an absolute Z-score greater than 3 are considered potential outliers. Outliers are also treated using the capping and flooring method by calculating the 5th and 95th percentiles and replacing values outside these limits with the corresponding boundary values. The salary distribution before and after outlier treatment is then compared using visualization. Since the dataset also contains categorical attributes such as `gender`, `hsc_s`, `degree_t`, and `specialisation`, these variables are converted into numerical representations using `LabelEncoder` and One-Hot Encoding. Finally, the completely transformed dataset is verified for missing values, data types, dimensions, and numerical representation, and the resulting model-ready dataset is saved as `Final.csv`. Thus, the experiment demonstrates the complete workflow of preparing real-world tabular data for data analysis and machine-learning applications.

## Step 1: Start by importing the necessary Python libraries for data preprocessing.


In [2]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl (8.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.18.0-cp314-cp314-macosx_14_0_arm64.whl (20.4 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn]0m 4/5 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [3]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
from scipy import stats


## Step 2: Load the placement dataset into a Pandas Dataframe.

In [8]:

df=pd.read_csv("data.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


In [9]:
df.shape

(215, 10)

In [10]:
df.head

<bound method NDFrame.head of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
3        4      M  52.00   Science       NaN   Sci&Tech     66.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
..     ...    ...    ...       ...       ...        ...      ...   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   
214    215      M  58.00   Science     53.00  Comm&Mgmt     89.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
3

In [11]:
df.tail

<bound method NDFrame.tail of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
3        4      M  52.00   Science       NaN   Sci&Tech     66.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
..     ...    ...    ...       ...       ...        ...      ...   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   
214    215      M  58.00   Science     53.00  Comm&Mgmt     89.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
3

In [12]:
df.sample(5)

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
44,45,F,73.0,Commerce,81.0,Comm&Mgmt,89.0,Mkt&Fin,69.70,200000.0
120,121,M,40.0,Science,59.0,Comm&Mgmt,73.0,Mkt&HR,58.81,NaN
68,69,F,47.0,Commerce,72.7,Sci&Tech,79.0,Mkt&HR,59.24,NaN
137,138,M,63.0,Commerce,72.0,Comm&Mgmt,56.0,Mkt&HR,60.41,225000.0
127,128,F,60.0,Science,69.0,Comm&Mgmt,55.5,Mkt&HR,58.40,250000.0


In [13]:
df.describe()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary
count,215.000000,210.000000,213.000000,211.000000,214.000000,148.000000
mean,108.000000,66.503000,66.420610,72.093649,62.254813,288655.405405
std,62.209324,10.904205,7.322786,13.340195,5.836962,93457.452420
min,1.000000,37.000000,50.000000,50.000000,51.210000,200000.000000
25%,54.500000,61.000000,61.000000,60.000000,57.922500,240000.000000
50%,108.000000,65.000000,66.000000,70.000000,61.950000,265000.000000
75%,161.500000,73.000000,72.000000,84.000000,66.187500,300000.000000
max,215.000000,97.700000,91.000000,98.000000,77.890000,940000.000000


In [14]:
df.loc[0]

sl_no                    1
gender                   M
hsc_p                 91.0
hsc_s             Commerce
degree_p              58.0
degree_t          Sci&Tech
etest_p               55.0
specialisation      Mkt&HR
mba_p                 58.8
salary            270000.0
Name: 0, dtype: object

In [15]:
df.iloc[0]

sl_no                    1
gender                   M
hsc_p                 91.0
hsc_s             Commerce
degree_p              58.0
degree_t          Sci&Tech
etest_p               55.0
specialisation      Mkt&HR
mba_p                 58.8
salary            270000.0
Name: 0, dtype: object

In [16]:
df[0:2]

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,91.00,Commerce,58.00,Sci&Tech,55.0,Mkt&HR,58.80,270000.0
1,2,M,78.33,Science,77.48,Sci&Tech,86.5,Mkt&Fin,66.28,200000.0


In [17]:
df["degree_p"]

0      58.00
1      77.48
2      64.00
3        NaN
4      73.30
       ...  
210    77.60
211    72.00
212    73.00
213    58.00
214    53.00
Name: degree_p, Length: 215, dtype: float64

## Step 3:Take a quick look at the data to understand its structure and identify any missing values or anomalies.

In [21]:
df.dropna(subset=["salary"],inplace=True)

In [22]:
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

#### The method isnull() checks each element in the DataFrame (or Series) to see if it is NaN (Not a Number) or None (missing value).
It returns a DataFrame (or Series) of the same shape as the input, with Boolean values:
#### True: The value is null (NaN or None).
#### False: The value is not null.

In [19]:
df.isnull().sum()


sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64

## Step 4: Handle Missing Data
### Option 1: If the dataset is large and only a small percentage of data is missing, you can remove rows with missing values using dropna(subset,inplace)


In [25]:
df["hsc_p"]=df["hsc_p"].fillna(df["hsc_p"].mean())

In [26]:
df["degree_p"]=df["degree_p"].fillna(df["degree_p"].mean())


In [27]:
df["etest_p"]=df["etest_p"].fillna(df["etest_p"].mean())

### Option 2:If removing data isn't ideal, you can impute (df.[""].fillna(df[""].mean(),inplace)) missing values using methods like mean, median, or most frequent.

## Step 5: Feature Scaling
Feature scaling is the process of converting numerical features to a similar scale so that one feature does not dominate another simply because it has larger numerical values.

<img src="https://i.postimg.cc/G21gMYnF/f.png" alt="Image Description" width="500">









## Option 1( StandardScaler): This method scales the data to have a mean of 0 and a standard deviation of 1.


In [29]:
c=["hsc_p","degree_p","etest_p","salary"]
s1=StandardScaler()
df[c]=s1.fit_transform(df[c])
df.head()



,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.341443e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


#### Option 2:This method scales the data to a fixed range, usually between 0 and 1. 
###  MinMaxScaler()

In [30]:
s2=MinMaxScaler()
df[c]=s2.fit_transform(df[c])
df.head()



,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 6  Option 1: Identifying Outliers Using Z-Scores
The value of 3 in the context of Z-scores is often used as a threshold to identify outliers in a dataset. A Z-score represents how many standard deviations a data point is away from the mean of the dataset. Specifically:

A Z-score of 0 means the data point is exactly at the mean.
A Z-score of 1 means the data point is one standard deviation above the mean, and so on.
A Z-score of 3 corresponds to a data point being 3 standard deviations away from the mean. For a normal distribution, about 99.7% of the data points fall within 3 standard deviations of the mean (according to the 68-95-99.7 rule, which describes the spread of data in a normal distribution). Therefore, points with Z-scores greater than 3 or less than -3 are considered unusually far from the mean and are often flagged as outliers.

This threshold (Z > 3 or Z < -3) is commonly used in many statistical applications because it captures the extreme values that are rare in a normal distribution, which are typically considered to be outliers. However, the choice of threshold can vary depending on the specific application and the nature of the data.



[![Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png](https://i.postimg.cc/2SFTtcXL/Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png)](https://postimg.cc/4Yyz75GX)

In [41]:
columns_to_check=["salary"]
zscore=stats.zscore(df[columns_to_check])
u=(zscore>3)
l=(zscore<-3)
print(u)
print(l)
indx=u | l
clean_df=df[~indx]
df.info
clean_df.info()

[[False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [ True]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [ True]
 [False]
 [False]
 

### Option 2:  Capping and Flooring Outliers
Capping and flooring is an outlier-treatment technique where extreme values are replaced with predefined boundary values instead of deleting the records.

In [45]:
l1=df["salary"].quantile(0.05)
u1=df["salary"].quantile(0.95)
df_capped=df.copy()
df_capped["salary"].clip(l1,u1)
df_capped.info()
df.head()


<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 12.7 KB


,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 7: Convert categorical variables into numerical format using LabelEncoder ().
[![Picture1.png](https://i.postimg.cc/yNpNvnVd/Picture1.png)](https://postimg.cc/zLW5fCHZ)




## Convert categorical variables into numerical format using one hot encoder

[![Picture2.png](https://i.postimg.cc/gcZ0Hv1J/Picture2.png)](https://postimg.cc/HjTHp7YD)

In [47]:
L1=LabelEncoder()
df["gender"]=L1.fit_transform(df["gender"])
df["hsc_s"]=L1.fit_transform(df["hsc_s"])
df["degree_t"]=L1.fit_transform(df["degree_t"])
df["specialisation"]=L1.fit_transform(df["specialisation"])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,1,0.857051,1,0.057143,2,0.104167,1,58.80,0.094595
1,2,1,0.586729,2,0.613714,2,0.760417,0,66.28,0.000000
2,3,1,0.409023,0,0.228571,0,0.520833,0,57.80,0.067568
4,5,1,0.485812,1,0.494286,0,0.975000,0,55.50,0.304054
7,8,1,0.280990,2,0.285714,2,0.354167,0,62.14,0.070270


In [51]:
c=["gender"]
one_hot_encoded_data=pd.get_dummies(df,columns=c)
one_hot_encoded_data.to_csv('clean.csv',index=False)
one_hot_encoded_data.head()

Y=df['salary'].copy()

# Remove target and ID from features
X=df.drop(columns=['salary','sl_no']).copy()

NameError: name 'sl_no' is not defined

# Exercise: Data Cleaning and Transformation – Automobile Dataset

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset
- Import Pandas and load the Automobile dataset.
- Display the first 10 rows.
- Display the shape of the dataset.

### 2. Explore the Dataset
- Display the column names.
- Display the data types.
- Generate descriptive statistics.
- Identify numerical and categorical columns.
- Display unique values in categorical columns.

## Step 2: Data Cleaning

### 3. Check Missing Values
- Check for missing values in each column.
- Display the number and percentage of missing values.

### 4. Handle Missing Values
- Replace missing numerical values using mean or median.
- Replace missing categorical values using mode.
- Verify that no missing values remain.

### 5. Remove Duplicate Records
- Check for duplicate rows.
- Display the number of duplicate records.
- Remove duplicate records.
- Verify the result.

### 6. Clean the `horsepower` Column
- Identify non-numeric values such as `?`.
- Replace `?` with `NaN`.
- Convert `horsepower` to numeric.
- Handle the resulting missing values.

## Step 3: Data Transformation

### 7. Transform the `origin` Column
- Display the unique values in `origin`.
- Convert the values into meaningful labels:
  - `1` → `usa`
  - `2` → `europe`
  - `3` → `japan`

### 8. Create `weight_kg`
- Create a new column `weight_kg`.
- Convert weight from pounds to kilograms.

  `weight_kg = weight × 0.453592`

### 9. Create `mpg_category`
Create a new column based on `mpg`:
- `< 20` → `Low`
- `20–29` → `Medium`
- `≥ 30` → `High`

### 10. Create `vehicle_age`
- Create a new column `vehicle_age`.
- Assume the current year is 2026.

  `vehicle_age = 2026 - model_year`

### 11. Rename Columns
Rename:
- `mpg` → `miles_per_gallon`
- `horsepower` → `hp`
- `weight` → `weight_lbs`
- `model_year` → `year`

### 12. Filter the Data
Display vehicles:
- With `mpg > 30`
- With `horsepower > 150`
- With `cylinders >= 6`
- Manufactured after 1980
- Originating from `usa`

## Step 4: Encoding Categorical Data

### 13. Label Encoding
- Apply `LabelEncoder` to the `origin` column.
- Create a new column `origin_encoded`.
- Display the original and encoded values.
- Display the category-to-label mapping.

### 14. One-Hot Encoding
- Apply One-Hot Encoding to the `origin` column.
- Compare Label Encoding and One-Hot Encoding.
- Which encoding method is more appropriate for `origin`? Explain why.

## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores
- Calculate the Z-score for the numerical features.
- Identify observations with `|Z-score| > 3` as outliers.
- Count the outliers in each numerical column.
- Display the rows containing outliers.
- Decide whether the outliers should be removed or retained.

## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler
- Select the numerical features.
- Apply `StandardScaler`.
- Display the standardized values.
- Verify that the features have approximately mean `0` and standard deviation `1`.

## Step 7: Normalization

### 17. Normalization Using MinMaxScaler
- Apply `MinMaxScaler` to the numerical features.
- Transform the features to the range `[0, 1]`.
- Display the normalized values.
- Compare **Standardization** and **Normalization**.
- Explain when each scaling method is appropriate.

## Step 8: Create Features and Target

### 18. Create X and Y Variables
- Select the appropriate input features as **X (independent variables)**.
- Select `mpg` as **Y (target variable)**.
- Display the shape of `X` and `Y`.
- Save `X` and `Y` into `automobile_X_Y.csv`.
- Load the CSV file again and display the first 5 rows.

## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset
- Combine the processed features and target variable.
- Display the final dataset.
- Check for missing values.
- Save the final dataset as `automobile_preprocessed.csv`.